# 06 - Independent Validation & Multi-Model Blind Evaluation

## Executive Summary & Methodology
A major challenge in concept-based explainability for Medical VLMs is **quantitative evaluation reliability**. Frameworks like *MedConcept* rely on Large Language Models as external evaluators (*LLM-as-a-judge*), but prior research highlights potential biases and report selectivity issues (*'Judging the Judges'*, Shi et al., 2025).

To address the feedback from Prof. Eleonora Poeta and guarantee scientific rigor, this notebook implements a **Double-Blind Independent Validation Protocol**.

### Core Design Principles of this Protocol:
1. **Blind Annotation (Bias Removal)**: The reference MedGemma verdicts are **hidden** from the exported spreadsheet to prevent anchoring bias for both human annotators and commercial LLMs.
2. **Dual-Format Testing (Text Prompt vs. File Attachment)**: Commercial models (ChatGPT GPT-4o, Claude 3.5 Sonnet, Gemini 1.5 Pro) are evaluated under two distinct input formats:
   - **File Attachment Mode** (`_file_verdict`): Uploading the structured CSV file directly into the model chat interface.
   - **Direct Batch Text Prompt Mode** (`_text_verdict`): Inpulting a formatted text prompt in a fresh chat session.
   - *Goal*: Measure format robustness and evaluate whether input modality affects judgment consistency.
3. **Human Multi-Rater Consensus**: Independent manual evaluation by the 3 project team members (Davide, Riccardo, Emmanuel) resolved by majority vote (`human_consensus`).
4. **Post-Evaluation Benchmarking**: MedGemma verdicts are merged back only during final statistical analysis to compute Inter-Annotator Agreement (IAA) and Cohen's / Fleiss' Kappa ($\kappa$).

## 1. Environment Setup and Imports

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for exact reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Set output directory
results_dir = os.path.join('..', 'results', '06_independent_validation')
os.makedirs(results_dir, exist_ok=True)
print(f"Output directory configured at: {os.path.abspath(results_dir)}")

## 2. Stratified Subset Extraction for Blind Validation

We load evaluation results from Notebook 04 (`df_evaluation_500_samples.csv` and details CSV) and extract a stratified subset of $N=25$ representative concept predictions covering `Aligned`, `Unaligned`, and `Uncertain` classes.

In [ ]:
# Path to evaluation results
eval_csv_path = os.path.join('..', 'results', '04_evaluation', 'df_evaluation_500_samples.csv')
details_csv_path = os.path.join('..', 'results', '04_evaluation', 'old', 'df_details_500_samples.csv')

if not os.path.exists(details_csv_path):
    details_csv_path = os.path.join('..', 'results', '04_evaluation', 'df_details_500_samples.csv')

df_details = pd.read_csv(details_csv_path)
df_eval = pd.read_csv(eval_csv_path)

# Merge report text into details
if 'report' not in df_details.columns:
    df_merged = pd.merge(df_details, df_eval[['image_index', 'report']], on='image_index', how='left')
else:
    df_merged = df_details.copy()

df_merged['medgemma_verdict'] = df_merged['verdict']
df_merged['predicted_concept'] = df_merged['concept']
df_merged['radiology_report'] = df_merged['report']

# Stratified Sampling (8 Aligned, 8 Unaligned, 9 Uncertain)
sample_size_per_class = {'Aligned': 8, 'Unaligned': 8, 'Uncertain': 9}
sampled_dfs = []

for verdict_cat, count in sample_size_per_class.items():
    subset = df_merged[df_merged['medgemma_verdict'] == verdict_cat]
    if len(subset) >= count:
        sampled = subset.sample(n=count, random_state=RANDOM_SEED)
    else:
        sampled = subset
    sampled_dfs.append(sampled)

df_validation_subset = pd.concat(sampled_dfs).reset_index(drop=True)
df_validation_subset['sample_id'] = [f"SAMPLE_{i+1:02d}" for i in range(len(df_validation_subset))]

print(f"Extracted Stratified Subset: {len(df_validation_subset)} samples.")
print("Ground Truth MedGemma Distribution (HIDDEN FROM BLIND EXPORT):")
print(df_validation_subset['medgemma_verdict'].value_counts())

## 3. Blind Spreadsheet Generation (`independent_validation_subset.csv`)

We generate a clean spreadsheet **without `medgemma_verdict`**. Columns are included for human annotators and dual-format LLM testing:

In [ ]:
# Build Blind Export DataFrame (medgemma_verdict is EXCLUDED)
df_blind_export = pd.DataFrame()
df_blind_export['sample_id'] = df_validation_subset['sample_id']
df_blind_export['radiology_report'] = df_validation_subset['radiology_report']
df_blind_export['predicted_concept'] = df_validation_subset['predicted_concept']

# Human Evaluation Columns
df_blind_export['davide_verdict'] = ''
df_blind_export['riccardo_verdict'] = ''
df_blind_export['emmanuel_verdict'] = ''
df_blind_export['human_consensus'] = ''

# Dual-Format LLM Evaluation Columns (File Upload vs Direct Text Prompt)
df_blind_export['chatgpt_file_verdict'] = ''
df_blind_export['chatgpt_text_verdict'] = ''
df_blind_export['claude_file_verdict'] = ''
df_blind_export['claude_text_verdict'] = ''
df_blind_export['gemini_file_verdict'] = ''
df_blind_export['gemini_text_verdict'] = ''
df_blind_export['notes'] = ''

csv_path = os.path.join(results_dir, 'independent_validation_subset.csv')
excel_path = os.path.join(results_dir, 'independent_validation_subset.xlsx')

df_blind_export.to_csv(csv_path, index=False)
print(f"Saved Blind CSV Spreadsheet to: {os.path.abspath(csv_path)}")

try:
    df_blind_export.to_excel(excel_path, index=False)
    print(f"Saved Blind Excel Spreadsheet to: {os.path.abspath(excel_path)}")
except Exception as e:
    print(f"Note: CSV saved cleanly. ({e})")

print("\nBlind Export Columns Preview:")
print(df_blind_export.columns.tolist())

## 4. Human Blind Annotation Guidelines (For Davide, Riccardo, and Emmanuel)

Annotate each sample independently in your column (`davide_verdict`, `riccardo_verdict`, or `emmanuel_verdict`) using these rules:

| Verdict Code | Name | Definition & Rule |
| :--- | :--- | :--- |
| **`1` / `Aligned`** | **Explicitly Present** | The report explicitly confirms or describes this finding/concept. |
| **`2` / `Unaligned`** | **Explicitly Contradicted** | The report explicitly denies or contradicts the concept. |
| **`3` / `Uncertain`** | **Unmentioned / Normal Structure** | The concept refers to standard unmentioned normal anatomy (e.g., *"ribs"*, *"aorta"*) OR is absent from the report. |


## 5. Dual-Format LLM Testing Protocols (File vs Direct Text Prompt)

### Protocol A: File Upload Mode (`_file_verdict`)
Attach `independent_validation_subset.csv` directly in the web chat of ChatGPT (GPT-4o), Claude 3.5 Sonnet, or Gemini 1.5 Pro and paste this prompt:

```text
You are an expert medical AI evaluator. Attached is a CSV file ('independent_validation_subset.csv') containing 25 samples of chest X-ray radiology reports paired with predicted visual concepts.

Task: For each sample (SAMPLE_01 to SAMPLE_25), evaluate the semantic alignment between the 'radiology_report' and the 'predicted_concept'.

Classification Rules:
- 'Aligned': The report explicitly mentions or confirms the concept.
- 'Unaligned': The report explicitly contradicts or denies the concept.
- 'Uncertain': The report does NOT mention the concept, OR the concept refers to unmentioned normal anatomy.

Output Format Requirement:
Return a clean CSV table with exactly two columns: 'sample_id' and 'verdict' (containing only Aligned, Unaligned, or Uncertain).
```

---

### Protocol B: Direct Text Prompt Mode (`_text_verdict`)
In a **new fresh chat session**, copy and paste the full batch text prompt generated below.

In [ ]:
# Generate Protocol B Batch Text Prompt
prompt_header = "You are an expert medical evaluator analyzing concept-based explanations in Vision-Language Models.\nEvaluate the semantic alignment between each 'Radiology Report' and the 'Predicted Concept'.\n\nCLASSIFICATION RULES:\n- Aligned: The report explicitly confirms or describes the predicted concept.\n- Unaligned: The report explicitly contradicts or denies the predicted concept.\n- Uncertain: The report does NOT mention the concept, OR it refers to unmentioned normal anatomy.\n\nSAMPLES TO EVALUATE:\n"
prompt_samples = []
for idx, row in df_blind_export.iterrows():
    sample_str = f"[{row['sample_id']}]\nReport: \"{row['radiology_report']}\"\nConcept: \"{row['predicted_concept']}\""
    prompt_samples.append(sample_str)

prompt_footer = "\n\nOUTPUT REQUIREMENT:\nProvide a markdown table with columns: | sample_id | verdict | reason |\nWhere verdict must be strictly one of: Aligned, Unaligned, Uncertain.\n"
full_batch_prompt = prompt_header + "\n\n".join(prompt_samples) + prompt_footer

prompt_file_path = os.path.join(results_dir, 'batch_llm_prompt.txt')
with open(prompt_file_path, 'w', encoding='utf-8') as pf:
    pf.write(full_batch_prompt)

print(f"Generated Direct Text Batch Prompt ({len(full_batch_prompt)} characters).")
print(f"Saved to: {os.path.abspath(prompt_file_path)}")

## 6. Inter-Annotator Agreement (IAA) & Format Consistency Analysis

Once the CSV file is populated, run the analysis below. It automatically merges `medgemma_verdict` back for benchmarking and computes:
1. **Human Consensus vs MedGemma**
2. **Format Consistency (File Upload vs Direct Text Prompt)** for ChatGPT, Claude, and Gemini
3. **Human Consensus vs Commercial LLM Variants**

In [ ]:
def analyze_blind_validation(file_path, ref_df):
    if not os.path.exists(file_path):
        print(f"File {file_path} not found.")
        return
    
    df_user = pd.read_csv(file_path)
    
    # Merge reference medgemma_verdict back for benchmarking
    df_anal = pd.merge(df_user, ref_df[['sample_id', 'medgemma_verdict']], on='sample_id', how='left')
    
    # Compute Human Consensus if not explicitly filled
    h_cols = [c for c in ['davide_verdict', 'riccardo_verdict', 'emmanuel_verdict'] if c in df_anal.columns]
    if h_cols and not df_anal[h_cols].isnull().all().all():
        df_anal['human_consensus'] = df_anal[h_cols].mode(axis=1)[0]
    
    print("=== 1. FORMAT CONSISTENCY ANALYSIS (FILE UPLOAD vs DIRECT TEXT PROMPT) ===")
    models = ['chatgpt', 'claude', 'gemini']
    for m in models:
        col_f = f"{m}_file_verdict"
        col_t = f"{m}_text_verdict"
        if col_f in df_anal.columns and col_t in df_anal.columns and not df_anal[col_f].isnull().all() and not df_anal[col_t].isnull().all():
            f_val = df_anal[col_f].astype(str).str.strip()
            t_val = df_anal[col_t].astype(str).str.strip()
            fmt_agree = (f_val == t_val).mean() * 100
            print(f"{m.upper()} Format Agreement (File vs Text): {fmt_agree:.2f}%")
    
    print("\n=== 2. BENCHMARK AGREEMENT AGAINST HUMAN CONSENSUS ===")
    if 'human_consensus' in df_anal.columns and not df_anal['human_consensus'].isnull().all():
        h_ref = df_anal['human_consensus'].astype(str).str.strip()
        test_cols = ['medgemma_verdict'] + [c for c in df_anal.columns if '_verdict' in c and 'davide' not in c and 'riccardo' not in c and 'emmanuel' not in c and 'human' not in c]
        for tc in test_cols:
            if tc in df_anal.columns and not df_anal[tc].isnull().all():
                pred = df_anal[tc].astype(str).str.strip()
                acc = (h_ref == pred).mean() * 100
                print(f"Human Consensus vs {tc.upper()}: {acc:.2f}%")
    else:
        print("Human consensus not populated yet. Please fill in the CSV file to see complete metrics.")

# Execute analysis preview
analyze_blind_validation(csv_path, df_validation_subset)